<a href="https://colab.research.google.com/github/marianoInsa/dimiasa-models/blob/main/notebooks/falls/pipeline/00_Preprocesamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline ETL de Preprocesamiento de Datasets de Caídas (100 Hz)

**Objetivo:** Pipeline de extracción, transformación y carga (ETL) para procesar los datos inmutables en la capa `bronce`, realizar la validación de integridad y filtrado de calidad de resampleo en la capa `plata`, y generar los archivos Parquet resampleados a 100 Hz en la capa `oro`.

### Arquitectura de Almacenamiento (Azure Data Lake)
* **`bronce/falls`**: CSVs crudos e inmutables de los datasets (`SisFall`, `FallAllD`, `KFall`, `UPFall`).
* **`plata/falls`**: Métricas de calidad por trial y configuraciones JSON de filtrado.
* **`oro/falls`**: Datasets finales en formato Parquet a 100 Hz con el esquema estandarizado de 7 columnas.

### Estándar de Unidades Físicas y Ejes
* **Acelerómetros (`Ax`, `Ay`, `Az`)**: Expresados en aceleración de la gravedad ($g$, donde $1g \approx 9.81\text{ m/s}^2$).
* **Giroscopios (`Gx`, `Gy`, `Gz`)**: Expresados en velocidad angular en grados por segundo ($\circ/\text{s}$).

### Esquema Final de Salida (Capa Oro)
`Subject`, `Activity_Label`, `Activity_Code`, `Trial`, `Sample_Index`, `AVM`, `GVM`

## 0 · Instalación de dependencias

In [1]:
%pip install numpy pandas scipy pyarrow dtaidistance azure-storage-file-datalake --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.8/287.8 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 435.6/435.6 kB 37.5 MB/s eta 0:00:00


## 1 · Constantes globales del pipeline

Todos los parámetros configurables del pipeline. Modificar estos valores antes de ejecutar si se desea ajustar umbrales o frecuencias.

In [2]:
import io
import json
import gc
from math import gcd
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy.signal import resample_poly, butter, filtfilt
from scipy.stats import pearsonr
from dtaidistance import dtw as dtw_lib
from google.colab import userdata
from azure.storage.filedatalake import DataLakeServiceClient

# --- Constantes configurables --------------------------------------------------
FS_TARGET         = 100      # Hz objetivo de resampleo
KAISER_BETA       = 5.0      # Parámetro de forma de la ventana Kaiser
EVAL_WINDOW_SEC   = 2.0      # Ventana de evaluación de métricas (segundos)

# Umbrales de descarte de calidad (lógica OR sobre AVM)
THR_PEARSON_MIN   = 0.85
THR_PHASE_MS_MAX  = 100.0
THR_ATTEN_PCT_MAX = 25.0

VALID_LABELS = {"Fall", "ADL"}

# Metadatos de datasets: nombre → frecuencia original y archivo CSV en bronce
DATASETS_META = {
    "UPFall":   {"fs": 100, "csv": "UPFall-Reduced.csv"},
    "KFall":    {"fs": 100, "csv": "KFall-Reduced.csv"},
    "FallAllD": {"fs": 238, "csv": "FallAllD-Reduced.csv"},
    "SisFall":  {"fs": 200, "csv": "SisFall-Reduced.csv"},
}

# Esquema de los CSV crudos (capa bronce)
RAW_SCHEMA_COLS = [
    "Subject", "Activity_Label", "Activity_Code", "Trial", "Sample_Index",
    "Ax", "Ay", "Az", "Gx", "Gy", "Gz",
]

# Esquema del Parquet final (capa oro): AVM y GVM resampleados
SCHEMA_COLS = [
    "Subject", "Activity_Label", "Activity_Code", "Trial", "Sample_Index",
    "AVM", "GVM",
]

SENSOR_COLS = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
META_COLS   = ["Subject", "Activity_Label", "Activity_Code", "Trial"]

print("Configuración del Pipeline ETL:")
print(f"  Frecuencia objetivo (FS_TARGET)      : {FS_TARGET} Hz")
print(f"  Parámetro ventana Kaiser (KAISER_BETA): {KAISER_BETA}")
print(f"  Umbral mínimo de Pearson r            : {THR_PEARSON_MIN}")
print(f"  Umbral máximo de desfase de pico      : {THR_PHASE_MS_MAX} ms")
print(f"  Umbral máximo de atenuación de pico   : {THR_ATTEN_PCT_MAX} %")
print(f"  Esquema final ({len(SCHEMA_COLS)} columnas): {SCHEMA_COLS}")

Configuración del Pipeline ETL:
  Frecuencia objetivo (FS_TARGET)      : 100 Hz
  Parámetro ventana Kaiser (KAISER_BETA): 5.0
  Umbral mínimo de Pearson r            : 0.85
  Umbral máximo de desfase de pico      : 100.0 ms
  Umbral máximo de atenuación de pico   : 25.0 %
  Esquema final (7 columnas): ['Subject', 'Activity_Label', 'Activity_Code', 'Trial', 'Sample_Index', 'AVM', 'GVM']


## 2 · Funciones del pipeline ETL

Definición completa de toda la lógica pura: resampleo, métricas de fidelidad, validación, filtrado y persistencia en Azure.

In [3]:
# --- Resampleo -----------------------------------------------------------------

def get_poly_factors(fs_orig: int, fs_target: int = FS_TARGET) -> tuple[int, int]:
    """Devuelve (up, down) reducidos al mínimo común divisor."""
    g = gcd(fs_target, fs_orig)
    return fs_target // g, fs_orig // g


def resample_signal(
    signal: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
    kaiser_beta: float = KAISER_BETA,
) -> np.ndarray:
    """Resamplea un vector 1D usando resample_poly y ventana Kaiser."""
    up, down = get_poly_factors(fs_orig, fs_target)
    return resample_poly(signal, up=up, down=down, window=("kaiser", kaiser_beta))


def resample_trial_df(
    trial_df: pd.DataFrame,
    fs_orig: int,
    fs_target: int = FS_TARGET,
    kaiser_beta: float = KAISER_BETA,
    schema_cols: list[str] | None = None,
) -> pd.DataFrame:
    """
    Calcula AVM/GVM desde las 6 señales crudas, resamplea ambas magnitudes y
    reconstruye el DataFrame final con el esquema de oro.
    Sample_Index se reinicia a 0..N-1 tras el resampleo.
    """
    if schema_cols is None:
        schema_cols = SCHEMA_COLS

    meta = {c: trial_df[c].iloc[0] for c in META_COLS if c in trial_df.columns}

    avm = np.sqrt(
        trial_df["Ax"].values.astype(float) ** 2
        + trial_df["Ay"].values.astype(float) ** 2
        + trial_df["Az"].values.astype(float) ** 2
    )
    gvm = np.sqrt(
        trial_df["Gx"].values.astype(float) ** 2
        + trial_df["Gy"].values.astype(float) ** 2
        + trial_df["Gz"].values.astype(float) ** 2
    )

    avm_r = resample_signal(avm, fs_orig, fs_target, kaiser_beta)
    gvm_r = resample_signal(gvm, fs_orig, fs_target, kaiser_beta)

    out = pd.DataFrame({"AVM": avm_r, "GVM": gvm_r})
    n_out = len(out)
    for c, val in meta.items():
        out[c] = val
    out["Sample_Index"] = np.arange(n_out)

    return out[schema_cols]


print("✓ Funciones de resampleo definidas.")

✓ Funciones de resampleo definidas.


In [4]:
# --- Métricas de fidelidad de resampleo ----------------------------------------

def lowpass(
    signal: np.ndarray,
    fs: int,
    cutoff: float | None = None,
    fs_target: int = FS_TARGET,
) -> np.ndarray:
    """
    Aplica filtro pasa-bajos Butterworth a la señal original para igualar el ancho
    de banda del objetivo antes de calcular métricas de fidelidad.
    """
    if cutoff is None:
        cutoff = fs_target / 2.0 - 0.5
    nyq = fs / 2.0
    if cutoff >= nyq * 0.99:
        return signal.copy()
    b, a = butter(8, cutoff / nyq, btype="low")
    return filtfilt(b, a, signal)


def snr_inband(
    orig: np.ndarray,
    resampled: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
) -> float:
    """Calcula SNR (dB) en la banda [0, FS_TARGET/2] Hz."""
    t_orig = np.linspace(0, 1, len(orig))
    t_res  = np.linspace(0, 1, len(resampled))
    resampled_interp = np.interp(t_orig, t_res, resampled)
    orig_filtered = lowpass(orig, fs_orig, fs_target=fs_target)
    noise = orig_filtered - resampled_interp
    power_signal = np.mean(orig_filtered ** 2)
    power_noise  = np.mean(noise ** 2)
    if power_noise == 0:
        return float("inf")
    return float(10 * np.log10(power_signal / power_noise))


def pearson_inband(
    orig: np.ndarray,
    resampled: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
) -> float:
    """Calcula correlación de Pearson entre original filtrada y resampleada."""
    t_orig = np.linspace(0, 1, len(orig))
    t_res  = np.linspace(0, 1, len(resampled))
    resampled_interp = np.interp(t_orig, t_res, resampled)
    orig_filtered = lowpass(orig, fs_orig, fs_target=fs_target)
    r, _ = pearsonr(orig_filtered, resampled_interp)
    return float(r)


def dtw_normalized(
    orig: np.ndarray,
    resampled: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
    kaiser_beta: float = KAISER_BETA,
) -> float:
    """Distancia DTW normalizada entre original resampleada y señal resampleada."""
    up, down = get_poly_factors(fs_orig, fs_target)
    orig_rs = resample_poly(orig, up=up, down=down, window=("kaiser", kaiser_beta))

    def zscore(x):
        s = x.std()
        return (x - x.mean()) / s if s > 0 else x - x.mean()

    a = zscore(orig_rs).astype(np.double)
    b = zscore(resampled).astype(np.double)
    if len(a) > 500:
        a, b = a[:500], b[:500]
    dist = dtw_lib.distance_fast(a, b)
    return float(dist / max(len(a), len(b)))


def peak_phase_shift_ms(
    orig: np.ndarray,
    resampled: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
) -> float:
    """Calcula desfase temporal del pico en milisegundos."""
    return float(abs(orig.argmax() / fs_orig - resampled.argmax() / fs_target) * 1000.0)


def peak_attenuation_pct(
    orig: np.ndarray,
    resampled: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
) -> float:
    """Calcula atenuación porcentual del pico relativo a la original filtrada."""
    orig_filtered = lowpass(orig, fs_orig, fs_target=fs_target)
    max_orig = np.max(orig_filtered)
    t_orig = np.linspace(0, 1, len(orig))
    t_res  = np.linspace(0, 1, len(resampled))
    max_res = np.max(np.interp(t_orig, t_res, resampled))
    if max_orig > 0:
        return float(abs(max_orig - max_res) / max_orig * 100.0)
    return 0.0


def analyze_trial(
    trial_df: pd.DataFrame,
    fs_orig: int,
    fs_target: int = FS_TARGET,
    kaiser_beta: float = KAISER_BETA,
    eval_window_sec: float = EVAL_WINDOW_SEC,
) -> dict | None:
    """Calcula métricas de fidelidad de resampleo para AVM y GVM de un trial."""
    avm = np.sqrt(
        trial_df["Ax"] ** 2 + trial_df["Ay"] ** 2 + trial_df["Az"] ** 2
    ).values
    gvm = np.sqrt(
        trial_df["Gx"] ** 2 + trial_df["Gy"] ** 2 + trial_df["Gz"] ** 2
    ).values

    if len(avm) < fs_orig:
        return None

    avm_r = resample_signal(avm, fs_orig, fs_target, kaiser_beta)
    gvm_r = resample_signal(gvm, fs_orig, fs_target, kaiser_beta)

    w      = int(eval_window_sec / 2.0 * fs_target)
    peak_r = avm_r.argmax()
    s, e   = max(0, peak_r - w), min(len(avm_r), peak_r + w)

    wo     = int(eval_window_sec / 2.0 * fs_orig)
    peak_o = avm.argmax()
    so, eo = max(0, peak_o - wo), min(len(avm), peak_o + wo)

    result = {}
    for sensor, orig, resampled in [
        ("AVM", avm[so:eo], avm_r[s:e]),
        ("GVM", gvm[so:eo], gvm_r[s:e]),
    ]:
        result[sensor] = {
            "snr_db":         snr_inband(orig, resampled, fs_orig, fs_target),
            "pearson_r":      pearson_inband(orig, resampled, fs_orig, fs_target),
            "dtw_norm":       dtw_normalized(orig, resampled, fs_orig, fs_target, kaiser_beta),
            "phase_shift_ms": peak_phase_shift_ms(orig, resampled, fs_orig, fs_target),
            "peak_atten_pct": peak_attenuation_pct(orig, resampled, fs_orig, fs_target),
        }
    return result


def run_trial_metrics(
    df: pd.DataFrame,
    fs_orig: int,
    fs_target: int = FS_TARGET,
    label: str = "Fall",
) -> pd.DataFrame:
    """Itera sobre todos los trials de la clase indicada y calcula métricas."""
    falls  = df[df["Activity_Label"] == label]
    trials = falls[["Subject", "Activity_Code", "Trial"]].drop_duplicates()
    rows   = []
    n_done = 0

    for _, row in trials.iterrows():
        mask = (
            (falls["Subject"]       == row["Subject"])
            & (falls["Activity_Code"] == row["Activity_Code"])
            & (falls["Trial"]         == row["Trial"])
        )
        res = analyze_trial(falls[mask], fs_orig, fs_target)
        n_done += 1
        if res is None:
            continue
        for sensor in ("AVM", "GVM"):
            rows.append({
                "Subject":       row["Subject"],
                "Activity_Code": row["Activity_Code"],
                "Trial":         row["Trial"],
                "sensor":        sensor,
                **res[sensor],
            })
        if n_done % 200 == 0 or n_done == len(trials):
            print(f"  Progreso: {n_done}/{len(trials)} trials", end="\r")
    print()
    return pd.DataFrame(rows)


print("✓ Funciones de métricas de fidelidad definidas.")

✓ Funciones de métricas de fidelidad definidas.


In [5]:
# --- Validación, filtrado de calidad y esquema de salida ----------------------

def validate_labels(
    df: pd.DataFrame, valid_labels: set[str] | None = None
) -> dict:
    """Verifica que Activity_Label contenga solo valores esperados y sin mezcla en trials."""
    if valid_labels is None:
        valid_labels = VALID_LABELS
    unexpected    = set(df["Activity_Label"].unique()) - valid_labels
    trial_counts  = (
        df.groupby(["Subject", "Activity_Code", "Trial"])["Activity_Label"].nunique()
    )
    return {
        "unexpected_labels": unexpected,
        "mixed_trials":      trial_counts[trial_counts > 1],
    }


def filter_valid_trials(
    df_raw: pd.DataFrame,
    df_metrics_avm: pd.DataFrame,
    pearson_min: float    = THR_PEARSON_MIN,
    phase_ms_max: float   = THR_PHASE_MS_MAX,
    atten_pct_max: float  = THR_ATTEN_PCT_MAX,
) -> tuple[list[list], int, int]:
    """
    Cruza trials de clase Fall con métricas de AVM y aplica umbrales de calidad.
    Retorna (valid_ids, n_valid, n_discarded).
    """
    falls    = df_raw[df_raw["Activity_Label"] == "Fall"]
    trial_ids = (
        falls[["Subject", "Activity_Code", "Trial"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    merged = trial_ids.merge(
        df_metrics_avm[["Subject", "Activity_Code", "Trial",
                        "pearson_r", "phase_shift_ms", "peak_atten_pct"]],
        on=["Subject", "Activity_Code", "Trial"],
        how="left",
    )
    discard_mask = (
        merged["pearson_r"].isna()
        | (merged["pearson_r"]      < pearson_min)
        | (merged["phase_shift_ms"] > phase_ms_max)
        | (merged["peak_atten_pct"] > atten_pct_max)
    )
    valid_ids   = merged.loc[~discard_mask, ["Subject", "Activity_Code", "Trial"]].values.tolist()
    n_discarded = int(discard_mask.sum())
    return valid_ids, len(valid_ids), n_discarded


def validate_schema(df: pd.DataFrame, schema_cols: list[str] | None = None) -> bool:
    """Verifica que el DataFrame tenga exactamente las columnas en el orden indicado."""
    if schema_cols is None:
        schema_cols = SCHEMA_COLS
    return list(df.columns) == schema_cols


print("✓ Funciones de validación y filtrado definidas.")

✓ Funciones de validación y filtrado definidas.


In [6]:
# --- Persistencia en Azure Data Lake (cliente inyectado) ----------------------

def load_csv_from_bronze(
    service_client,
    container_name: str = "bronce",
    directory_name: str = "falls",
    filename: str = "SisFall-Reduced.csv",
) -> pd.DataFrame:
    """Descarga un CSV desde la capa bronce de Azure Data Lake."""
    content = (
        service_client
        .get_file_system_client(container_name)
        .get_directory_client(directory_name)
        .get_file_client(filename)
        .download_file()
        .readall()
    )
    return pd.read_csv(io.BytesIO(content))


def _to_native(obj):
    """Convierte recursivamente tipos numpy a tipos Python nativos para JSON."""
    if isinstance(obj, dict):         return {k: _to_native(v) for k, v in obj.items()}
    if isinstance(obj, list):         return [_to_native(i) for i in obj]
    if isinstance(obj, np.integer):   return int(obj)
    if isinstance(obj, np.floating):  return float(obj)
    return obj


def _get_file_client(service_client, container_name: str, directory_name: str, filename: str):
    return (
        service_client
        .get_file_system_client(container_name)
        .get_directory_client(directory_name)
        .get_file_client(filename)
    )


def save_parquet_to_gold(
    service_client,
    df: pd.DataFrame,
    ds_name: str,
    container_name: str = "oro",
    directory_name: str = "falls",
) -> int:
    """Valida el esquema y exporta un DataFrame Parquet a la capa oro."""
    if not validate_schema(df, SCHEMA_COLS):
        raise ValueError(
            f"El esquema del DataFrame no coincide con SCHEMA_COLS. "
            f"Columnas actuales: {list(df.columns)}"
        )
    buf = io.BytesIO()
    df.to_parquet(buf, index=False, engine="pyarrow")
    parquet_bytes = buf.getvalue()
    fc = _get_file_client(service_client, container_name, directory_name, f"{ds_name}.parquet")
    fc.upload_data(parquet_bytes, overwrite=True, length=len(parquet_bytes))
    return len(parquet_bytes)


def save_json_to_silver(
    service_client,
    data: dict,
    filename: str,
    container_name: str = "plata",
    directory_name: str = "falls",
) -> int:
    """Serializa un diccionario a JSON y lo sube a la capa plata."""
    json_bytes = json.dumps(_to_native(data), indent=2, ensure_ascii=False).encode("utf-8")
    fc = _get_file_client(service_client, container_name, directory_name, filename)
    fc.upload_data(json_bytes, overwrite=True, length=len(json_bytes))
    return len(json_bytes)


def save_csv_to_silver(
    service_client,
    df: pd.DataFrame,
    filename: str,
    container_name: str = "plata",
    directory_name: str = "falls",
) -> int:
    """Guarda un DataFrame en CSV y lo sube a la capa plata."""
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    csv_bytes = buf.getvalue().encode("utf-8")
    fc = _get_file_client(service_client, container_name, directory_name, filename)
    fc.upload_data(csv_bytes, overwrite=True, length=len(csv_bytes))
    return len(csv_bytes)


print("✓ Funciones de persistencia en Azure definidas.")

✓ Funciones de persistencia en Azure definidas.


## 3 · Conexión a Azure Data Lake e Ingesta desde Capa Bronce

Se establece la conexión con el Data Lake utilizando las credenciales seguras del entorno de Colab.

In [7]:
CONNECTION_STRING = userdata.get("cadenaAzure")
service_client    = DataLakeServiceClient.from_connection_string(CONNECTION_STRING)

raw_datasets: dict[str, pd.DataFrame] = {}

print("Descargando datasets crudos desde bronce/falls...")
for ds_name, meta in DATASETS_META.items():
    try:
        df = load_csv_from_bronze(
            service_client,
            container_name="bronce",
            directory_name="falls",
            filename=meta["csv"]
        )
        raw_datasets[ds_name] = df
        print(f"  ✓ {ds_name:10s} ({meta['fs']} Hz) → {len(df):>10,} filas cargadas.")
    except Exception as err:
        print(f"  ✗ Error al cargar {ds_name}: {err}")

print(f"\nDatasets listos en memoria: {list(raw_datasets.keys())}")

Descargando datasets crudos desde bronce/falls...
  ✓ UPFall     (100 Hz) →    294,678 filas cargadas.
  ✓ KFall      (100 Hz) →  3,995,100 filas cargadas.
  ✓ FallAllD   (238 Hz) →  8,558,480 filas cargadas.
  ✓ SisFall    (200 Hz) → 15,858,929 filas cargadas.

Datasets listos en memoria: ['UPFall', 'KFall', 'FallAllD', 'SisFall']


## 4 · Validación de Integridad de Etiquetas

Verifica que la columna `Activity_Label` contenga únicamente los valores válidos (`Fall` y `ADL`) y detecta trials anómalos con mezcla incoherente de etiquetas.

In [8]:
label_summary_rows = []

print("Validando integridad de etiquetas...")
for ds_name, df in raw_datasets.items():
    res = validate_labels(df)

    if res["unexpected_labels"]:
        print(f"  ⚠️ [{ds_name}] Etiquetas inesperadas: {res['unexpected_labels']}")
    else:
        print(f"  ✅ [{ds_name}] Etiquetas válidas exclusivamente (Fall/ADL).")

    if len(res["mixed_trials"]) > 0:
        print(f"  ⚠️ [{ds_name}] {len(res['mixed_trials'])} trial(s) con mezcla incoherente.")
    else:
        print(f"  ✅ [{ds_name}] Sin mezcla de etiquetas en trials.")

    trial_labels = df.groupby(["Subject", "Activity_Code", "Trial"])["Activity_Label"].first()
    n_fall = (trial_labels == "Fall").sum()
    n_adl  = (trial_labels == "ADL").sum()
    label_summary_rows.append({
        "Dataset": ds_name, "ADL": n_adl, "Fall": n_fall,
        "Total Trials": n_fall + n_adl,
        "Ratio ADL/Fall": round(n_adl / n_fall, 2) if n_fall > 0 else 0.0
    })

print("\nDistribución inicial de trials por dataset:")
print(pd.DataFrame(label_summary_rows).set_index("Dataset").to_string())

Validando integridad de etiquetas...
  ✅ [UPFall] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [UPFall] Sin mezcla de etiquetas en trials.
  ✅ [KFall] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [KFall] Sin mezcla de etiquetas en trials.
  ✅ [FallAllD] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [FallAllD] Sin mezcla de etiquetas en trials.
  ✅ [SisFall] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [SisFall] Sin mezcla de etiquetas en trials.

Distribución inicial de trials por dataset:
           ADL  Fall  Total Trials  Ratio ADL/Fall
Dataset                                           
UPFall     304   255           559            1.19
KFall     2729  2346          5075            1.16
FallAllD  1332   466          1798            2.86
SisFall   2707  1798          4505            1.51


## 5 · Generación de Métricas de Fidelidad por Trial (Capa Plata)

Para los datasets con tasa de muestreo original distinta a 100 Hz (`SisFall` @200Hz y `FallAllD` @238Hz), se calculan las cinco métricas de fidelidad de resampleo sobre AVM y GVM. Los resultados se persisten en `plata/falls/resampling_metrics_per_trial_100hz.csv`.

In [9]:
metrics_results: dict[str, pd.DataFrame] = {}
metrics_frames = []

for ds_name, meta in DATASETS_META.items():
    if ds_name not in raw_datasets:
        continue

    fs_orig = meta["fs"]
    if fs_orig == FS_TARGET:
        print(f"[{ds_name}] {fs_orig} Hz = objetivo. Sin resampleo real, se omiten métricas.")
        continue

    print(f"\n[{ds_name}] Calculando métricas de fidelidad ({fs_orig} Hz → {FS_TARGET} Hz)...")
    df_mets = run_trial_metrics(raw_datasets[ds_name], fs_orig=fs_orig, fs_target=FS_TARGET)
    metrics_results[ds_name] = df_mets
    metrics_frames.append(df_mets.assign(Dataset=ds_name))
    print(f"  ✓ {len(df_mets):,} filas de métricas generadas.")

df_all_metrics = pd.concat(metrics_frames, ignore_index=True) if metrics_frames else pd.DataFrame()

if not df_all_metrics.empty:
    csv_filename  = f"resampling_metrics_per_trial_{FS_TARGET}hz.csv"
    bytes_written = save_csv_to_silver(
        service_client, df_all_metrics, csv_filename, "plata", "falls"
    )
    print(f"\n✅ {csv_filename} → plata/falls/ ({bytes_written:,} bytes, {len(df_all_metrics):,} filas).")

[UPFall] 100 Hz = objetivo. Sin resampleo real, se omiten métricas.
[KFall] 100 Hz = objetivo. Sin resampleo real, se omiten métricas.

[FallAllD] Calculando métricas de fidelidad (238 Hz → 100 Hz)...
  Progreso: 466/466 trials
  ✓ 932 filas de métricas generadas.

[SisFall] Calculando métricas de fidelidad (200 Hz → 100 Hz)...
  Progreso: 1798/1798 trials
  ✓ 3,596 filas de métricas generadas.

✅ resampling_metrics_per_trial_100hz.csv → plata/falls/ (506,088 bytes, 4,528 filas).


## 6 · Filtrado de Calidad por Trial y Persistencia de Configuración

Se aplican los umbrales de descarte sobre la magnitud vectorial de aceleración (AVM):
- Pearson $r \ge 0.85$
- Desfase de pico $\le 100\text{ ms}$
- Atenuación de pico $\le 25\%$

Se sube `trial_quality_config.json` a `plata/falls/` con la lista de IDs de trials válidos por dataset.

In [10]:
quality_config = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "fs_target": FS_TARGET,
    "criteria": {
        "pearson_r_min":      THR_PEARSON_MIN,
        "phase_shift_ms_max": THR_PHASE_MS_MAX,
        "peak_atten_pct_max": THR_ATTEN_PCT_MAX,
    },
    "datasets": {}
}

print("Filtrando trials según criterios de calidad de resampleo:\n")
print(f"  {'Dataset':12s}  {'Total Fall':>10}  {'Válidos':>9}  {'Descartados':>12}  {'% Descarte':>10}")
print("  " + "-" * 58)

for ds_name, meta in DATASETS_META.items():
    if ds_name not in raw_datasets:
        continue

    df_raw = raw_datasets[ds_name]
    falls  = df_raw[df_raw["Activity_Label"] == "Fall"]
    total_falls = falls[["Subject", "Activity_Code", "Trial"]].drop_duplicates().shape[0]

    if ds_name not in metrics_results:
        # KFall / UPFall: sin resampleo real, todos los trials son válidos
        valid_ids   = falls[["Subject", "Activity_Code", "Trial"]].drop_duplicates().values.tolist()
        n_valid, n_discarded = total_falls, 0
    else:
        df_avm = metrics_results[ds_name]
        df_avm = df_avm[df_avm["sensor"] == "AVM"].reset_index(drop=True)
        valid_ids, n_valid, n_discarded = filter_valid_trials(
            df_raw, df_avm,
            pearson_min=THR_PEARSON_MIN,
            phase_ms_max=THR_PHASE_MS_MAX,
            atten_pct_max=THR_ATTEN_PCT_MAX
        )

    pct_discard = (n_discarded / total_falls * 100) if total_falls > 0 else 0.0
    quality_config["datasets"][ds_name] = {
        "total": total_falls, "valid": n_valid,
        "discarded": n_discarded, "valid_ids": valid_ids,
    }
    print(f"  {ds_name:12s}  {total_falls:>10,}  {n_valid:>9,}  {n_discarded:>12,}  {pct_discard:>9.1f}%")

json_bytes = save_json_to_silver(
    service_client, quality_config, "trial_quality_config.json", "plata", "falls"
)
print(f"\n✅ trial_quality_config.json → plata/falls/ ({json_bytes:,} bytes).")

Filtrando trials según criterios de calidad de resampleo:

  Dataset       Total Fall    Válidos   Descartados  % Descarte
  ----------------------------------------------------------
  UPFall               255        255             0        0.0%
  KFall              2,346      2,346             0        0.0%
  FallAllD             466        417            49       10.5%
  SisFall            1,798      1,704            94        5.2%

✅ trial_quality_config.json → plata/falls/ (337,457 bytes).


## 7 · Resampleo Definitivo a 100 Hz y Exportación a Capa Oro

Por cada dataset se calcula AVM/GVM, se resamplea con filtro Kaiser ($\beta=5.0$) y se exporta con el esquema de 7 columnas:
`Subject`, `Activity_Label`, `Activity_Code`, `Trial`, `Sample_Index`, `AVM`, `GVM`

Los archivos Parquet se guardan en `oro/falls/{dataset}.parquet`.

In [11]:
print("Procesando resampleo definitivo y exportación a Parquet en oro/falls/:\n")

for ds_name in list(DATASETS_META.keys()):
    if ds_name not in raw_datasets or ds_name not in quality_config["datasets"]:
        continue

    fs_orig  = DATASETS_META[ds_name]["fs"]
    df_raw   = raw_datasets.pop(ds_name)
    gc.collect()

    valid_set = {
        (row[0], row[1], row[2])
        for row in quality_config["datasets"][ds_name]["valid_ids"]
    }

    df_fall = df_raw[df_raw["Activity_Label"] == "Fall"]
    df_adl  = df_raw[df_raw["Activity_Label"] == "ADL"]
    df_fall_valid = df_fall[
        df_fall.apply(
            lambda r: (r["Subject"], r["Activity_Code"], r["Trial"]) in valid_set, axis=1
        )
    ]

    df_to_resample = pd.concat([df_fall_valid, df_adl], ignore_index=True)
    del df_raw, df_fall, df_adl, df_fall_valid
    gc.collect()

    resampled_chunks = []
    for _, grp in df_to_resample.groupby(["Subject", "Activity_Code", "Trial"], sort=False):
        grp_copy = grp.copy()
        for col in SENSOR_COLS:
            if col in grp_copy.columns:
                grp_copy[col] = grp_copy[col].astype("float32")
        resampled_chunks.append(
            resample_trial_df(
                grp_copy, fs_orig=fs_orig, fs_target=FS_TARGET,
                kaiser_beta=KAISER_BETA, schema_cols=SCHEMA_COLS
            )
        )

    del df_to_resample
    gc.collect()

    df_out = pd.concat(resampled_chunks, ignore_index=True)
    del resampled_chunks
    gc.collect()

    if not validate_schema(df_out, SCHEMA_COLS):
        raise ValueError(f"[{ds_name}] El esquema final no coincide con el orden esperado de 7 columnas.")

    n_fall_rows = (df_out["Activity_Label"] == "Fall").sum()
    n_adl_rows  = (df_out["Activity_Label"] == "ADL").sum()

    bytes_uploaded = save_parquet_to_gold(
        service_client, df_out, ds_name, "oro", "falls"
    )

    print(f"  ✓ [{ds_name:10s}] {fs_orig} Hz → {FS_TARGET} Hz")
    print(f"    Fall: {n_fall_rows:>9,} filas | ADL: {n_adl_rows:>9,} filas | Total: {len(df_out):>9,}")
    print(f"    ✅ Exportado a oro/falls/{ds_name}.parquet ({bytes_uploaded / (1024**2):.2f} MB)\n")

    del df_out
    gc.collect()

print("=" * 65)
print("  Pipeline ETL finalizado exitosamente. Capa Oro lista para uso.")
print("=" * 65)

Procesando resampleo definitivo y exportación a Parquet en oro/falls/:

  ✓ [UPFall    ] 100 Hz → 100 Hz
    Fall:    45,951 filas | ADL:   248,727 filas | Total:   294,678
    ✅ Exportado a oro/falls/UPFall.parquet (4.02 MB)

  ✓ [KFall     ] 100 Hz → 100 Hz
    Fall: 1,725,407 filas | ADL: 2,269,693 filas | Total: 3,995,100
    ✅ Exportado a oro/falls/KFall.parquet (59.81 MB)

  ✓ [FallAllD  ] 238 Hz → 100 Hz
    Fall:   834,000 filas | ADL: 2,664,000 filas | Total: 3,498,000
    ✅ Exportado a oro/falls/FallAllD.parquet (56.15 MB)

  ✓ [SisFall   ] 200 Hz → 100 Hz
    Fall: 2,555,917 filas | ADL: 5,232,748 filas | Total: 7,788,665
    ✅ Exportado a oro/falls/SisFall.parquet (127.73 MB)

  Pipeline ETL finalizado exitosamente. Capa Oro lista para uso.


## 8 · Evidencia y Justificación Técnica de la Frecuencia Objetivo (100 Hz)

Resumen estadístico de las métricas de fidelidad de señal calculadas internamente durante la ejecución del pipeline sobre AVM:

In [12]:
if not df_all_metrics.empty:
    avm_mets = df_all_metrics[df_all_metrics["sensor"] == "AVM"]
    summary_list = []
    for ds_name, grp in avm_mets.groupby("Dataset"):
        summary_list.append({
            "Dataset":                     ds_name,
            "Pearson r (Mediana)":          grp["pearson_r"].median(),
            "Pearson r (P05)":              grp["pearson_r"].quantile(0.05),
            "Desfase pico (Mediana ms)":    grp["phase_shift_ms"].median(),
            "Desfase pico (P95 ms)":        grp["phase_shift_ms"].quantile(0.95),
            "Atenuación pico (Mediana %)": grp["peak_atten_pct"].median(),
            "Atenuación pico (P95 %)":     grp["peak_atten_pct"].quantile(0.95),
        })
    print("Resumen numérico de fidelidad de señal a 100 Hz (AVM):")
    print(pd.DataFrame(summary_list).set_index("Dataset").round(3).to_string())
else:
    print("No se requirió resampleo (todos los datasets tienen frecuencia nativa a 100 Hz).")

Resumen numérico de fidelidad de señal a 100 Hz (AVM):
          Pearson r (Mediana)  Pearson r (P05)  Desfase pico (Mediana ms)  Desfase pico (P95 ms)  Atenuación pico (Mediana %)  Atenuación pico (P95 %)
Dataset                                                                                                                                               
FallAllD                0.973            0.479                        0.0                    0.0                        4.674                   13.667
SisFall                 0.990            0.845                        0.0                    0.0                        3.724                   13.252


### Conclusión
Las métricas obtenidas confirman que **100 Hz** es la frecuencia de muestreo adecuada:
1. Preserva la forma de la onda ($\text{mediana de } r \ge 0.90$).
2. Minimiza el desplazamiento temporal del instante de impacto (desfase P95 $\le 100\text{ ms}$).
3. Controla la pérdida de amplitud por filtrado anti-aliasing (atenuación P95 $\le 25\%$).